In [ ]:
# --- 必要なパッケージの読み込み ---
library(caret)
library(Metrics)
library(ggplot2)
library(monmlp)
library(lattice)
library(dplyr)

# --- OOD分割関数 (変更なし) ---
create_ood_split <- function(df, feature_cols, ood_ratio = 0.1) {
    df_features <- df[, feature_cols, drop = FALSE]
    dist_matrix <- dist(df_features, method = "euclidean")
    avg_distances <- as.matrix(dist_matrix) %>% colMeans()
    num_ood <- floor(nrow(df) * ood_ratio)
    if (num_ood == 0 && nrow(df) > 0) num_ood <- 1 
    ood_indices <- order(avg_distances, decreasing = TRUE)[1:num_ood]
    test_set <- df[ood_indices, ]
    train_set <- df[-ood_indices, ]
    cat(paste0("  OOD Split: Total=", nrow(df), ", Train=", nrow(train_set), ", Test(OOD)=", nrow(test_set), "\n"))
    return(list(train = train_set, test = test_set, ood_indices = ood_indices))
}

# --- 設定 ---
file_names <- c(
    "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
    "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
    "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
    "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

# 【重要】除外する列のリスト (IDとして特定された X を指定)
excluded_cols <- c("X") 

base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_hidden1", "Best_n.ensemble", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features", "OOD_R2", "OOD_RMSE") 

result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# 軸スケール決定関数
get_axis_limit <- function(values) {
    max_val <- max(values, na.rm = TRUE)
    if (max_val <= 1.5) { return(ceiling(max_val / 0.1) * 0.1)
    } else if (max_val <= 5) { return(ceiling(max_val / 0.5) * 0.5)
    } else if (max_val <= 30) { return(ceiling(max_val / 2) * 2)
    } else { return(ceiling(max_val / 5) * 5) }
}

#--- メインループ ---#
for (file in file_names) {
    filepath <- paste0(base_path, file)
    cat("\n=== Processing:", file, " (Excluding ID: X) ===\n")
    
    # 1. データの読み込み
    df_all <- read.csv(filepath)
    
    # 【重要修正】ID列（X）が存在すれば即座に削除
    df_all <- df_all[, setdiff(colnames(df_all), excluded_cols), drop = FALSE]
    
    feature_vars_initial <- setdiff(colnames(df_all), target_vars)

    for (target_var in target_vars) {
        cat("\n--- Training monmlp for:", target_var, " ---\n")
        
        # 欠損除去と数値限定
        df_temp <- df_all[, c(feature_vars_initial, target_var)]
        df_temp <- df_temp[complete.cases(df_temp), ]
        df_temp <- df_temp[, sapply(df_temp, is.numeric)]
        
        # ゼロ分散変数の除去
        nzv <- nearZeroVar(df_temp, saveMetrics = TRUE)
        df_complete <- df_temp[, !(nzv$zeroVar | nzv$nzv)]
        
        feature_vars_final <- setdiff(colnames(df_complete), target_var)

        if (nrow(df_complete) < 20 || length(feature_vars_final) == 0) {
            cat("  Skipping: insufficient data.\n")
            next
        }

        # === 2. OOD分割 (Xを含まない純粋な物理・構造空間で行う) ===
        is_fp_data <- grepl("_FP\\.csv$", file) 
        if (is_fp_data) {
            split_list <- create_ood_split(df_complete, feature_vars_final, ood_ratio = 0.1) 
            df_train_cv <- split_list$train
            df_ood_test <- split_list$test
        } else {
            df_train_cv <- df_complete
            df_ood_test <- NULL
        }

        # === 3. monmlpモデルの訓練 ===
        tune_grid <- expand.grid(hidden1 = c(2, 4, 6), n.ensemble = c(1, 3, 5))
        ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

        model <- train(
            formula(paste(target_var, "~ .")),
            data      = df_train_cv,
            method    = "monmlp",
            metric    = "RMSE",
            trControl = ctrl,
            tuneGrid  = tune_grid,
            preProcess = c("center", "scale")
        )

        # === 4. CV結果 (内挿性能) の計算と格納 ===
        cv_preds <- model$pred
        best_params <- model$bestTune
        for (param in names(best_params)) {
             cv_preds <- cv_preds[cv_preds[[param]] == best_params[[param]], ]
        }

        pred <- cv_preds$pred
        obs <- cv_preds$obs

        R2 <- round(cor(obs, pred)^2, 3)
        RMSE_val <- round(rmse(obs, pred), 3)
        MAE_val <- round(mae(obs, pred), 3)
        RPD_val <- round(sd(obs) / RMSE_val, 3)

        result_matrix[paste0("Best_hidden1_", target_var), file] <- model$bestTune$hidden1
        result_matrix[paste0("Best_n.ensemble_", target_var), file] <- model$bestTune$n.ensemble
        result_matrix[paste0("R2_", target_var), file] <- R2
        result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
        result_matrix[paste0("MAE_", target_var), file] <- MAE_val
        result_matrix[paste0("RPD_", target_var), file] <- RPD_val
        result_matrix[paste0("n_samples_", target_var), file] <- nrow(df_complete)
        result_matrix[paste0("n_features_", target_var), file] <- length(feature_vars_final)

        # === 5. OODテスト性能 (外挿性能) の計算 ===
        if (is_fp_data && !is.null(df_ood_test) && nrow(df_ood_test) > 0) {
            ood_preds <- predict(model, newdata = df_ood_test)
            ood_obs <- df_ood_test[[target_var]]
            ood_R2 <- round(cor(ood_obs, ood_preds)^2, 3)
            ood_RMSE_val <- round(rmse(ood_obs, ood_preds), 3)
            result_matrix[paste0("OOD_R2_", target_var), file] <- ood_R2
            result_matrix[paste0("OOD_RMSE_", target_var), file] <- ood_RMSE_val
        } else {
            result_matrix[paste0("OOD_R2_", target_var), file] <- NA
            result_matrix[paste0("OOD_RMSE_", target_var), file] <- NA
        }

        # === 6. モデルとプロットの保存 (fixed_ を付与) ===
        model_data_bundle <- list(model = model, training_data = df_train_cv, ood_test_data = df_ood_test)
        rds_file <- paste0("fixed20250702_model_data_monmlp_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".rds")
        saveRDS(model_data_bundle, file = rds_file)

        # プロット作成
        range_max <- get_axis_limit(c(obs, pred))
        p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
            geom_point(color = "darkred", alpha = 0.6) +
            geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "blue") +
            scale_x_continuous(limits = c(0, range_max)) +
            scale_y_continuous(limits = c(0, range_max)) +
            coord_fixed(ratio = 1) +
            labs(title = paste0(target_var, " (monmlp - ", file, ")"), 
                 subtitle = paste0("R2=", R2, " RPD=", RPD_val), x = "Observed", y = "Predicted") +
            theme_minimal()

        plot_file <- paste0("fixed20250702_plot_monmlp_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".pdf")
        pdf(plot_file, width = 6, height = 6)
        print(p)
        dev.off()
    }
}

#--- 結果保存 ---#
output_file <- paste0("fixed20250702_monmlp_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\n✅ Fixed monmlp Summary saved as:", output_file, "\n")